<a href="https://colab.research.google.com/github/rohini-th/Machine-Learning-project/blob/main/PCA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Principal Component Analysis

### Build a model that predicts the car Weight on basis of different properties of the car including car's manufacturer and model details

In [2]:
path=r"https://raw.githubusercontent.com/sindhura-nk/Datasets/refs/heads/main/Cars93.csv"
import pandas as pd
df=pd.read_csv(path,na_values=['NA',''],keep_default_na=False)
df.head()

,id,Manufacturer,Model,Type,Min.Price,Price,Max.Price,MPG.city,MPG.highway,AirBags,...,Passengers,Length,Wheelbase,Width,Turn.circle,Rear.seat.room,Luggage.room,Weight,Origin,Make
0,1,Acura,Integra,Small,12.9,15.9,18.8,25,31,None,...,5,177,102,68,37,26.5,11.0,2705,non-USA,Acura Integra
1,2,Acura,Legend,Midsize,29.2,33.9,38.7,18,25,Driver & Passenger,...,5,195,115,71,38,30.0,15.0,3560,non-USA,Acura Legend
2,3,Audi,90,Compact,25.9,29.1,32.3,20,26,Driver only,...,5,180,102,67,37,28.0,14.0,3375,non-USA,Audi 90
3,4,Audi,100,Midsize,30.8,37.7,44.6,19,26,NaN,...,6,193,106,70,37,31.0,17.0,3405,non-USA,Audi 100
4,5,BMW,535i,Midsize,23.7,30.0,36.2,22,30,Driver only,...,4,186,109,69,39,27.0,13.0,3640,non-USA,BMW 535i


### Perform basic data quality checks

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 93 entries, 0 to 92
Data columns (total 28 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id                  93 non-null     int64  
 1   Manufacturer        93 non-null     object 
 2   Model               93 non-null     object 
 3   Type                93 non-null     object 
 4   Min.Price           93 non-null     float64
 5   Price               93 non-null     float64
 6   Max.Price           93 non-null     float64
 7   MPG.city            93 non-null     int64  
 8   MPG.highway         93 non-null     int64  
 9   AirBags             89 non-null     object 
 10  DriveTrain          93 non-null     object 
 11  Cylinders           93 non-null     object 
 12  EngineSize          93 non-null     float64
 13  Horsepower          93 non-null     int64  
 14  RPM                 93 non-null     int64  
 15  Rev.per.mile        93 non-null     int64  
 16  Man.trans.

In [4]:
df.duplicated().sum()

np.int64(0)

In [5]:
df=df.drop_duplicates()

In [6]:
df.isna().sum()

,0
id,0
Manufacturer,0
Model,0
Type,0
Min.Price,0
Price,0
Max.Price,0
MPG.city,0
MPG.highway,0
AirBags,4


### Seperate X and Y features

In [7]:
X=df.drop(columns='Weight')
Y=df[['Weight']]

In [9]:
X.shape[1]

27

### Seperate Training and Testing Features

In [10]:
from sklearn.model_selection import train_test_split
xtrain,xtest,ytrain,ytest=train_test_split(X,Y,train_size=0.75,random_state=21)


### Data Cleaning and Data Scaling

In [11]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.compose import ColumnTransformer

cat = list(X.select_dtypes(include='object').columns)
con = list(X.select_dtypes(include='number').columns)

num_pipe = make_pipeline(
    SimpleImputer(strategy='median'),
    StandardScaler()
)

cat_pipe = make_pipeline(
    SimpleImputer(strategy='most_frequent'),
    OneHotEncoder(handle_unknown='ignore',sparse_output=False)
)

pre = ColumnTransformer([
    ('cat',cat_pipe,cat),
    ('con',num_pipe,con)
]).set_output(transform='pandas')

In [12]:
pre.fit(xtrain)

ColumnTransformer(transformers=[('cat',
                                 Pipeline(steps=[('simpleimputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehotencoder',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 ['Manufacturer', 'Model', 'Type', 'AirBags',
                                  'DriveTrain', 'Cylinders', 'Man.trans.avail',
                                  'Origin', 'Make']),
                                ('con',
                                 Pipeline(steps=[('simpleimputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('standardscaler',
                                                  StandardScaler())]),
                                 ['id', 'Min.Price', 'Price', 'Max.Price',
                                  'MPG.city', 'MPG.highway', 'EngineSize',
                                  'Horsepower', 'RPM', 'Rev.per.mile',
                                  'Fuel.tank.capacity', 'Passengers', 'Length',
                                  'Wheelbase', 'Width', 'Turn.circle',
                                  'Rear.seat.room', 'Luggage.room'])])

In [13]:
xtrain_pre=pre.transform(xtrain)
xtest_pre=pre.transform(xtest)

In [14]:
xtrain_pre.head()

,cat__Manufacturer_Acura,cat__Manufacturer_Audi,cat__Manufacturer_BMW,cat__Manufacturer_Buick,cat__Manufacturer_Cadillac,cat__Manufacturer_Chevrolet,cat__Manufacturer_Chrysler,cat__Manufacturer_Dodge,cat__Manufacturer_Eagle,cat__Manufacturer_Ford,...,con__RPM,con__Rev.per.mile,con__Fuel.tank.capacity,con__Passengers,con__Length,con__Wheelbase,con__Width,con__Turn.circle,con__Rear.seat.room,con__Luggage.room
49,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.246600,0.363774,1.199156,-0.983974,0.660629,0.380037,0.513278,0.032028,-1.068151,-1.812835
37,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,-1.748133,-1.877152,1.019218,0.876141,2.158675,1.592641,2.363378,1.294845,0.846198,2.721991
14,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,-0.084393,0.537727,-0.030424,0.876141,1.159978,0.683188,0.513278,0.347732,0.271893,0.832480
39,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.248355,1.878190,-1.260005,-0.983974,-1.265430,-0.984143,-0.543922,-0.599381,-1.259586,-1.057031
68,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,-0.084393,0.476332,-0.030424,-0.053916,0.589293,0.228462,0.248978,0.979141,0.080458,0.832480


### Create the principal components using PCA

In [15]:
from sklearn.decomposition import PCA
pca=PCA(n_components=2)
pca.fit(xtrain_pre)

PCA(n_components=2)

In [16]:
X_pca=pca.transform(xtrain_pre)
X_pca

array([[ 1.93170353,  4.18153439],
       [ 5.54853134, -1.91336033],
       [ 0.79142706, -1.76830987],
       [-4.03681394,  0.35322006],
       [ 0.23829195, -0.99377479],
       [-1.85212774, -0.92155361],
       [-2.37920492,  0.18434017],
       [-0.17873841, -1.80691947],
       [-0.64880298,  0.07957685],
       [ 3.2018932 ,  0.13988314],
       [ 2.61381873, -1.12480113],
       [ 6.96052086, -2.07619641],
       [ 0.86919613,  1.82214636],
       [-1.01993418,  0.85777439],
       [-0.62124822,  0.42890367],
       [ 2.05467854, -0.35987104],
       [ 4.21739518,  5.91604212],
       [-1.15082322, -1.74825576],
       [ 2.59921408, -1.27355849],
       [-7.31994247, -0.30033777],
       [ 5.73888246,  4.47658964],
       [-0.21658603, -0.34876415],
       [-5.6708416 , -0.75917499],
       [-0.60933908,  0.093036  ],
       [-0.86735236,  0.78294002],
       [ 1.49093021,  0.4957385 ],
       [-4.44719556, -0.5617944 ],
       [-3.66752823, -0.90187767],
       [-0.84537034,

## Create new data using the principal components

In [17]:
X_new=pd.DataFrame(X_pca)
X_new.head()

,0,1
0,1.931704,4.181534
1,5.548531,-1.913360
2,0.791427,-1.768310
3,-4.036814,0.353220
4,0.238292,-0.993775


In [18]:
X_new.columns=['PC1','PC2']
X_new.head()

,PC1,PC2
0,1.931704,4.181534
1,5.548531,-1.913360
2,0.791427,-1.768310
3,-4.036814,0.353220
4,0.238292,-0.993775


In [20]:
X_new.shape[1]

2

In [23]:
xnewtrain,xnewtest,ynewtrain,ynewtest = train_test_split(X_new,ytrain,train_size=0.75,random_state=21)

## Model Building

In [27]:
from sklearn.ensemble import RandomForestRegressor
rn = RandomForestRegressor(n_estimators=50,
                           max_depth=5)
rn.fit(xnewtrain,ynewtrain['Weight'])




RandomForestRegressor(max_depth=5, n_estimators=50)

In [28]:
rn.score(xnewtrain,ynewtrain)

0.983057971993469

In [29]:
rn.score(xnewtest,ynewtest)

0.9384281804266928